
# Demo Orbit Viewer — Física corregida

Este notebook corrige la construcción de órbitas del demo para usar propagación Kepleriana **two-body** consistente.

Se divide en dos partes:

1. **Generalización reusable** para cualquier cuerpo (desde estado heliocéntrico de Horizons).
2. **Aplicación específica a Apophis** y estimación de fecha de máxima cercanía Tierra–Apophis.


In [ ]:

# Instalar dependencias (omitir si ya están instaladas)
import sys
!{sys.executable} -m pip install -q pymcel plotly numpy pandas ipywidgets


## 1) Generalización reusable para cualquier cuerpo

In [ ]:

import numpy as np
import pandas as pd
import pymcel as pc
from dataclasses import dataclass
from datetime import datetime, timedelta

from orbit_viewer import OrbitElements, propagate_two_body, sample_times, make_orbit_viewer

MU_SUN = 2.9591220828559093e-4  # AU^3 / day^2
J2000 = datetime(2000, 1, 1, 12, 0, 0)


def date_to_j2000_days(date_str: str) -> float:
    dt = datetime.fromisoformat(date_str)
    return (dt - J2000).total_seconds() / 86400.0


def j2000_days_to_datetime(days: float) -> datetime:
    return J2000 + timedelta(days=float(days))


def _first_horizons_row(data):
    """Compatibilidad para salida de pymcel.consulta_horizons."""
    table = data[0] if isinstance(data, (tuple, list)) else data
    return table[0]


def fetch_heliocentric_state(body_id: str, epoch_iso: str):
    """Estado [x,y,z,vx,vy,vz] heliocéntrico en AU y AU/day."""
    data = pc.consulta_horizons(
        id=body_id,
        location='@10',  # heliocéntrico (Sol)
        epochs=epoch_iso,
        datos='vectors',
        propiedades='default',
    )
    row = _first_horizons_row(data)
    x, y, z = float(row['x']), float(row['y']), float(row['z'])
    vx, vy, vz = float(row['vx']), float(row['vy']), float(row['vz'])
    return np.array([x, y, z, vx, vy, vz], dtype=float)


def true_to_mean_anomaly_deg(nu_deg: float, e: float) -> float:
    nu = np.deg2rad(nu_deg)
    E = 2 * np.arctan2(np.sqrt(1 - e) * np.sin(nu / 2), np.sqrt(1 + e) * np.cos(nu / 2))
    M = E - e * np.sin(E)
    return float(np.rad2deg(M) % 360.0)


def elements_from_state(state_xyzvxyz: np.ndarray, epoch_days: float, mu: float = MU_SUN) -> OrbitElements:
    """Convierte estado cartesiano -> OrbitElements compatible con orbit_viewer."""
    p, e, inc_deg, raan_deg, argp_deg, nu_deg = pc.estado_a_elementos(mu, state_xyzvxyz)
    a = p / (1 - e**2)
    M0_deg = true_to_mean_anomaly_deg(nu_deg, e)
    return OrbitElements(
        a=float(a),
        e=float(e),
        i=float(inc_deg),
        raan=float(raan_deg),
        argp=float(argp_deg),
        M0=float(M0_deg),
        epoch=float(epoch_days),
        mu=float(mu),
        angle_unit='deg',
        distance_unit='au',
        time_unit='day',
        frame='ecliptic',
    )


@dataclass
class BodyOrbit:
    name: str
    body_id: str
    elements: OrbitElements
    orbit_positions: np.ndarray
    epoch_position: np.ndarray


def build_body_orbit(body_id: str, name: str, epoch_iso: str = '2028-01-01', n_orbit_points: int = 900) -> BodyOrbit:
    """Pipeline general para cualquier cuerpo: Horizons -> elementos -> órbita two-body."""
    epoch_days = date_to_j2000_days(epoch_iso)
    state = fetch_heliocentric_state(body_id, epoch_iso)
    elements = elements_from_state(state, epoch_days, mu=MU_SUN)
    times_orbit = sample_times(epoch_days, epoch_days + elements.period(), n_orbit_points)
    orbit_positions = propagate_two_body(elements, times_orbit)
    epoch_position = state[:3].copy()
    return BodyOrbit(
        name=name,
        body_id=body_id,
        elements=elements,
        orbit_positions=orbit_positions,
        epoch_position=epoch_position,
    )


In [ ]:

# Ejemplo de uso genérico (descomentar para cualquier cuerpo)
# cuerpo = build_body_orbit(body_id='499', name='Marte', epoch_iso='2028-01-01')
# print(cuerpo)


## 2) Aplicación a Apophis (corregida y comparada con nuestros modelos)

In [ ]:

# IDs Horizons
BODY_IDS = {
    'Mercurio': '199',
    'Venus': '299',
    'Tierra': '399',
    'Marte': '499',
    'Apophis': '99942',
}

EPOCH_ISO = '2028-01-01'

print('Construyendo órbitas two-body desde estados heliocéntricos de Horizons...')
mercurio = build_body_orbit(BODY_IDS['Mercurio'], 'Mercurio', EPOCH_ISO)
venus = build_body_orbit(BODY_IDS['Venus'], 'Venus', EPOCH_ISO)
tierra = build_body_orbit(BODY_IDS['Tierra'], 'Tierra', EPOCH_ISO)
marte = build_body_orbit(BODY_IDS['Marte'], 'Marte', EPOCH_ISO)
apophis = build_body_orbit(BODY_IDS['Apophis'], '99942 Apophis', EPOCH_ISO)
print('Listo.')


In [ ]:

reference_bodies = [
    {'name': 'Mercurio', 'positions': mercurio.orbit_positions, 'color': '#ff80c0', 'epoch_pos': mercurio.epoch_position},
    {'name': 'Venus',    'positions': venus.orbit_positions,    'color': '#9040d0', 'epoch_pos': venus.epoch_position},
    {'name': 'Tierra',   'positions': tierra.orbit_positions,   'color': '#40a0ff', 'epoch_pos': tierra.epoch_position},
    {'name': 'Marte',    'positions': marte.orbit_positions,    'color': '#ff5500', 'epoch_pos': marte.epoch_position},
]

fig = make_orbit_viewer(
    body_name='99942 Apophis',
    body_positions=apophis.orbit_positions,
    body_elements=apophis.elements,
    body_color='#e0e0e0',
    body_epoch_pos=apophis.epoch_position,
    reference_bodies=reference_bodies,
    epoch_label=f'epoch: {EPOCH_ISO} 00:00:00 UTC',
    title='Orbit Viewer — Apophis (física two-body corregida)',
    n_ticks=90,
    show_wedge=True,
)
fig.show()


In [ ]:

# Ventana de interés usada en los modelos del proyecto
T_START = date_to_j2000_days('2028-01-01')
T_END   = date_to_j2000_days('2029-07-01')
ts = np.linspace(T_START, T_END, 20000)

r_earth = propagate_two_body(tierra.elements, ts)
r_apoph = propagate_two_body(apophis.elements, ts)
d_au = np.linalg.norm(r_apoph - r_earth, axis=1)

imin = int(np.argmin(d_au))
tmin = float(ts[imin])
fecha_min = j2000_days_to_datetime(tmin)
dmin_au = float(d_au[imin])
dmin_km = dmin_au * 149_597_870.7

# Referencia de modelos acertados del repo (~2029-04-13)
fecha_ref = datetime(2029, 4, 13, 18, 10, 0)
error_horas = (fecha_min - fecha_ref).total_seconds() / 3600.0

print('Máxima cercanía Tierra–Apophis (two-body heliocéntrico):')
print(f'  fecha mínima = {fecha_min:%Y-%m-%d %H:%M:%S} UTC')
print(f'  distancia mínima = {dmin_au:.6e} AU = {dmin_km:,.0f} km')
print('Referencia modelos N-cuerpos del proyecto: 2029-04-13 ~18:10 UTC')
print(f'  diferencia temporal vs referencia = {error_horas:+.2f} h')


In [ ]:

import plotly.graph_objects as go

t_datetimes = [j2000_days_to_datetime(t) for t in ts]

fig_d = go.Figure()
fig_d.add_trace(go.Scatter(x=t_datetimes, y=d_au, mode='lines', name='d(Tierra,Apophis) [AU]'))
fig_d.add_vline(x=fecha_min, line_dash='dash', line_color='white')
fig_d.add_vline(x=datetime(2029, 4, 13, 18, 10, 0), line_dash='dot', line_color='yellow')
fig_d.update_layout(
    title='Distancia Tierra–Apophis (two-body) en 2028–2029',
    xaxis_title='Fecha UTC',
    yaxis_title='Distancia [AU]',
    template='plotly_dark',
)
fig_d.show()
